# REASON-X: Adaptive Test-Time Compute via Utility-Surface Regression

**Student Author:** Shreyas Kumar (23BAI1008)  
**Corresponding Author & Supervisor:** Dr. Rishikeshan C. A. (rishikeshan.ca@vit.ac.in), School of Computer Science and Engineering (SCOPE), Vellore Institute of Technology (VIT), Chennai  
**Comparative Baseline:** Zhai et al., *AdaCompute: Adaptive Test-Time Compute Allocation* (arXiv:2604.14853)  

---

### Abstract & Overview
This notebook contains the complete experimental pipeline and empirical validation for **REASON-X**, an adaptive test-time compute allocation framework for Large Language Models (LLMs). While traditional adaptive methods restrict allocation decisions to varying sample budgets under fixed Self-Consistency (1-D action space), REASON-X extends allocation to a joint topology-budget space:
$$\mathcal{A} = \{ \text{SC}, \text{Best-of-}N, \text{Tree-of-Thought (ToT)}, \text{Forest-of-Thought (FoT)} \} \times \{ b \in \mathcal{B} \}$$

To overcome the sample sparsity and label-collapse issues inherent in training multi-class discrete classifiers over combinatorial action spaces, REASON-X introduces a **Utility-Surface Regressor** $f_\theta(x, a) \approx \mathrm{Acc}(x, a)$ coupled with a constrained **Lagrangian Dual Allocator**.

### Experimental Setup
- **Hardware:** Dual NVIDIA Tesla T4 GPUs (16 GB VRAM each).
- **Benchmark Suite:** 200 mathematical reasoning instances sampled uniformly across:
  - **MATH-500** ($N=100$, LaTeX format, competitive mathematics)
  - **GSM8K** ($N=100$, numeric format, multi-step arithmetic)
- **Evaluated Checkpoints:** `Qwen/Qwen2.5-3B-Instruct` and `microsoft/Phi-3.5-mini-instruct`.
- **Reproducibility:** All stage artifacts, utility tensors $\mathbf{U} \in [0, 1]^{N \times |\mathcal{A}|}$, cost tensors $\mathbf{C} \in \mathbb{R}_{+}^{N \times |\mathcal{A}|}$, and checkpoints are serialized and mirrored to Hugging Face (`shkshreyas/reason-x-ckpt`).


In [1]:
import os, sys, subprocess, time
T_BOOT = time.time()

# Multi-GPU memory management: configure classic allocator to avoid peer-to-peer CUDA errors on dual T4
try:
    import torch
    if torch.cuda.device_count() <= 1:
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    else:
        # Two GPUs: expandable_segments is off (see above). These two options use the
        # classic allocator, so they don't trigger that error. max_split_size_mb stops
        # large cached blocks being carved up for small tensors, and the GC threshold
        # returns unused cached blocks before the card fills.
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
            "garbage_collection_threshold:0.8,max_split_size_mb:512")
except Exception:
    os.environ.pop("PYTORCH_CUDA_ALLOC_CONF", None)

os.environ.update({"TOKENIZERS_PARALLELISM": "false"})
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers>=4.44", "accelerate", "peft", "datasets",
                "scikit-learn", "xgboost"],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import json, math, random, re, itertools, warnings, gc, threading, pickle
from dataclasses import dataclass, field
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
from queue import Queue
import numpy as np, pandas as pd, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import builtins, functools
print = functools.partial(builtins.print, flush=True)   # never sit on a buffered log
warnings.filterwarnings("ignore")

N_GPU = torch.cuda.device_count()
CAP = torch.cuda.get_device_capability(0) if N_GPU else (0, 0)
DTYPE = torch.bfloat16 if CAP[0] >= 8 else torch.float16      # T4 = sm_75, fp16 only
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")
DEV  = "cuda:0" if N_GPU else "cpu"
DUAL = (N_GPU >= 2)
# Each worker owns one card and runs a whole model on it, acting as its own critic.
# Two workers then generate at the same time, instead of one card generating while
# the other waits to verify.
T0 = time.time(); el = lambda: f"{(time.time()-T0)/60:.1f} min"
STATS = {"tok": 0, "sec": 0.0}
print(f"dtype={DTYPE} | {N_GPU} GPU(s) | one model per card: {DUAL}")


  cuda:0  Tesla T4  15.6 GB  sm_75
  cuda:1  Tesla T4  15.6 GB  sm_75
dtype=torch.float16 | 2 GPU(s) | one model per card: True


## 1. Experimental Configuration, Model Checkpoints, and Benchmark Ingestion

Configures runtime profiles, evaluation models, random seeds, and loads MATH-500 and GSM8K test sets.


In [2]:
PROFILE = "FULL"           # <<<< "SMOKE" | "SMALL" | "FULL"

# Primary solvers evaluated throughout the paper (Table 1-5, Fig 3-6):
PAPER_MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-Math-1.5B-Instruct",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
]

# Extended solvers for 3B+ scale evaluation / 5-model study:
EXT_MODELS = [
    "Qwen/Qwen2.5-3B-Instruct",            # larger, same family as Qwen2.5-1.5B
    "microsoft/Phi-3.5-mini-instruct",     # 3.8B, a different model family
]

# Active evaluation models for dual-GPU run:
MODELS = EXT_MODELS

@dataclass
class Cfg:
    seed: int = 1008
    critic_id: str = "Qwen/Qwen2.5-1.5B-Instruct"
    n_models: int = len(MODELS)
    n_math: int = 130; n_gsm: int = 70
    n_sample: int = 16
    budgets: tuple = (1, 2, 4, 8, 16)
    max_new: int = 400
    probe_tok: int = 64; probe_k: int = 4
    temp: float = 0.7; top_p: float = 0.95
    batch_tok: int = 24_000; batch_seq: int = 48
    critic_tok: int = 28_000      # lowered automatically on OOM
    targets: tuple = (1.5, 2.0, 3.0)
    tot_beam: int = 3; tot_depth: int = 3; fot_trees: int = 3
    tree_runs: int = 3           # Paper Section 6: each tree action averaged over 3 runs
    alpha: float = 0.15; delta: float = 0.20
    n_seeds: int = 5
    reserve: float = 0.20
    resume: bool = True          # reuse per-model checkpoints on restart
    parallel: bool = False       # one model at a time: in parallel the models fight over
                                 # Kaggle's few CPU cores (3B: 208 min alone vs 514 in parallel)
    out: str = "./rx_out"

CFG = Cfg()

if PROFILE == "SMOKE":
    CFG.n_math, CFG.n_gsm, CFG.n_sample = 6, 4, 4
    CFG.budgets = (1, 2, 4); CFG.tree_runs = 1
elif PROFILE == "SMALL":
    CFG.n_math, CFG.n_gsm, CFG.tree_runs = 26, 14, 1

# Kaggle/Colab output directory
if os.path.isdir("/kaggle/working"): CFG.out = "/kaggle/working/rx_out"
elif os.path.isdir("/content"):     CFG.out = "/content/rx_out"
os.makedirs(f"{CFG.out}/figs", exist_ok=True); os.makedirs(f"{CFG.out}/ckpt", exist_ok=True)

# Restore checkpoints from previous run if attached as input
import glob, shutil
_found = [f for f in glob.glob("/kaggle/input/**/*.pkl", recursive=True)
          if f"_{CFG.n_math+CFG.n_gsm}q" in os.path.basename(f)]
for f in _found:
    dst = f"{CFG.out}/ckpt/{os.path.basename(f)}"
    if not os.path.exists(dst): shutil.copy(f, dst)
if _found: print(f"restored {len(_found)} checkpoint file(s) from a previous run")

# ── Off-Kaggle checkpoint backup (private Hugging Face dataset) ──────────────────
# Every checkpoint is uploaded the moment it is written and pulled back at start-up,
# so a session that ends (12-hour limit, crash, closed browser) loses nothing finished.
HF_REPO = "shkshreyas/reason-x-ckpt"

# Preferred: a Kaggle secret named HF_TOKEN (Add-ons -> Secrets, and TICK it for this
# notebook). Fallback: paste the token between the quotes below. Only do that if the
# notebook stays PRIVATE, because saved versions keep the code; revoke it afterwards.
HF_TOKEN_MANUAL = ""

def _hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t and t.strip(): return t.strip(), "Kaggle secret"
    except Exception as e:
        print(f"  (Kaggle secret HF_TOKEN not readable: {type(e).__name__}: {e})")
    if os.environ.get("HF_TOKEN", "").strip(): return os.environ["HF_TOKEN"].strip(), "environment"
    if HF_TOKEN_MANUAL.strip(): return HF_TOKEN_MANUAL.strip(), "pasted in cell 2"
    return None, None

HF_TOKEN, _src = _hf_token()
SYNC = False
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN      # also authenticates the model and dataset downloads
    try:
        from huggingface_hub import HfApi, snapshot_download
        _hf = HfApi(token=HF_TOKEN)
        _who = _hf.whoami()["name"]
        _hf.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
        SYNC = True
        print(f"backup ON  -> huggingface.co/datasets/{HF_REPO} (logged in as {_who}, token from {_src})")
    except Exception as e:
        print(f"backup OFF: a token was found ({_src}) but Hugging Face rejected it -> "
              f"{type(e).__name__}: {e}\n  Check it is a WRITE token and was copied in full.")
else:
    print("backup OFF: no token found. Add the Kaggle secret HF_TOKEN and tick it for this "
          "notebook, or paste the token into HF_TOKEN_MANUAL.")
if SYNC:
    try:
        _d = snapshot_download(HF_REPO, repo_type="dataset", token=HF_TOKEN,
                               local_dir=f"{CFG.out}/_hf_pull")
        _n = 0
        for f in glob.glob(f"{_d}/**/*.pkl", recursive=True):
            dst = f"{CFG.out}/ckpt/{os.path.basename(f)}"
            if not os.path.exists(dst): shutil.copy(f, dst); _n += 1
        print(f"  pulled {_n} checkpoint file(s) from the backup")
    except Exception as e:
        print(f"  nothing pulled from the backup yet ({type(e).__name__})")

def sync_up(path):
    """Upload one checkpoint in the background. Never blocks or crashes the run."""
    if not SYNC: return
    def _go():
        try:
            _hf.upload_file(path_or_fileobj=path, path_in_repo=f"ckpt/{os.path.basename(path)}",
                            repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN,
                            commit_message=f"checkpoint {os.path.basename(path)}")
            print(f"  [backup] saved {os.path.basename(path)}")
        except Exception as e:
            print(f"  [backup] upload FAILED for {os.path.basename(path)}: {e}")
    threading.Thread(target=_go, daemon=True).start()
random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
print("models:", [m.split("/")[-1] for m in MODELS])
print(f"target: {CFG.n_math} MATH + {CFG.n_gsm} GSM8K = {CFG.n_math+CFG.n_gsm} questions")

ANS_RE = re.compile(r"(-?[\d,]*\.?\d+)")
def norm_num(s):
    s = str(s).strip().replace(",", "").replace("$", "").replace("%", "").rstrip(".")
    try:
        v = float(s); return str(int(v)) if abs(v-round(v)) < 1e-6 else f"{v:.6f}".rstrip("0").rstrip(".")
    except Exception: return s.lower().strip()
def norm_tex(s):
    s = str(s)
    s = re.sub(r"\\left|\\right|\\!|\\,|\\;|\\ ", "", s)
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    s = s.replace("^{\\circ}", "").replace("^\\circ", "").replace("\\%", "")
    s = s.replace("\\$", "").replace("$", "").replace(" ", "")
    s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
    return s.rstrip(".").strip()
def boxed(t):
    i = t.rfind("\\boxed")
    if i < 0: return None
    j = t.find("{", i)
    if j < 0: return None
    d, k = 1, j+1
    while k < len(t) and d:
        d += (t[k]=="{") - (t[k]=="}")
        k += 1
    return t[j+1:k-1] if d == 0 else None
def extract(text, kind):
    t = text.strip()
    m = re.findall(r"(?:final answer|answer)\s*[::]\s*(.+)", t, flags=re.I)
    cand = m[-1].strip().split("\n")[0].strip().strip(".") if m else None
    b = boxed(t)
    if b is not None: cand = b
    if kind == "num":
        g = ANS_RE.findall((cand if cand else t).replace(",", ""))
        return norm_num(g[-1]) if g else None
    return norm_tex(cand) if cand else None
def grade(p, gold, kind):
    if p is None: return 0
    if kind == "num": return int(norm_num(p) == norm_num(gold))
    a, b_ = norm_tex(p), norm_tex(gold)
    if a == b_: return 1
    try: return int(abs(float(a)-float(b_)) < 1e-6)
    except Exception: return 0
kf = lambda kind: (norm_num if kind == "num" else norm_tex)
def majority(preds, kind):
    v = [p for p in preds if p is not None]
    if not v: return None, 0.0
    c = Counter(kf(kind)(x) for x in v); top, n = c.most_common(1)[0]
    for x in v:
        if kf(kind)(x) == top: return x, n/len(preds)
    return v[0], n/len(preds)

@dataclass
class Item:
    qid: str; task: str; kind: str; question: str; gold: str; level: int = 0

from datasets import load_dataset
def load_all(cfg):
    out = []
    try:
        ds = [r for r in load_dataset("HuggingFaceH4/MATH-500", split="test")]
        random.Random(cfg.seed).shuffle(ds)
        for i, r in enumerate(ds[:cfg.n_math]):
            d = re.findall(r"\d", str(r.get("level", "3")))
            out.append(Item(f"math-{i}", "math", "latex", r["problem"].strip(),
                            str(r["answer"]).strip(), int(d[0]) if d else 3))
    except Exception as e: print("MATH-500 unavailable ->", e)
    ds = [r for r in load_dataset("openai/gsm8k", "main", split="test")]
    random.Random(cfg.seed).shuffle(ds)
    for i, r in enumerate(ds[:cfg.n_gsm]):
        out.append(Item(f"gsm-{i}", "gsm8k", "num", r["question"].strip(),
                        norm_num(r["answer"].split("####")[-1].strip()), 0))
    random.Random(cfg.seed+1).shuffle(out)
    return out

ITEMS = load_all(CFG)
QIDS = [it.qid for it in ITEMS]
BYID = {it.qid: it for it in ITEMS}
print(f"target: {CFG.n_math} MATH + {CFG.n_gsm} GSM8K = {CFG.n_math+CFG.n_gsm} questions")
print(f"{len(ITEMS)} questions | {dict(Counter(x.task for x in ITEMS))}")


backup ON  -> huggingface.co/datasets/shkshreyas/reason-x-ckpt (logged in as shkshreyas, token from Kaggle secret)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

  pulled 15 checkpoint file(s) from the backup
models: ['Qwen2.5-3B-Instruct', 'Phi-3.5-mini-instruct']
target: 130 MATH + 70 GSM8K = 200 questions


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

...

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

target: 130 MATH + 70 GSM8K = 200 questions
200 questions | {'gsm8k': 70, 'math': 130}


## 2. Reasoning Topologies, Verifier Guidance, and Feature Extractors

Defines core inference primitives: Self-Consistency (SC), Best-of-$N$ (BoN), Tree-of-Thought (ToT), Forest-of-Thought (FoT), and entropy feature extraction.


In [3]:
from transformers import (AutoModelForCausalLM, AutoTokenizer, LogitsProcessor,
                          LogitsProcessorList, StoppingCriteria, StoppingCriteriaList)

# Compatibility check for PyTorch OutOfMemoryError across versions
OOMErr = getattr(torch, "OutOfMemoryError", None) or torch.cuda.OutOfMemoryError

@torch.inference_mode()
def _sanity_check(m, tk, mid):
    """Stop immediately if a freshly loaded model cannot answer a trivial question.
    Costs ~2 s and catches any silently broken weights before hours of generation."""
    msgs = [{"role": "user", "content": "What is 17 + 25? Reply with just the number."}]
    try:    p = tk.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception: p = msgs[0]["content"]
    enc = tk(p, return_tensors="pt").to(m.device)
    o = m.generate(**enc, max_new_tokens=16, do_sample=False, pad_token_id=tk.pad_token_id)
    ans = tk.decode(o[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    assert "42" in ans, (f"{mid} FAILED the post-load sanity check (answered {ans!r}). "
                         "Its weights did not load correctly; stopping before wasting GPU time.")
    print(f"  {mid.split('/')[-1]} sanity check passed: {ans.strip()[:24]!r}")

def load_lm(mid, dev):
    if dev.startswith("cuda"):
        # Explicitly set device context per worker thread
        torch.cuda.set_device(int(dev.split(":")[1]))
    tk = AutoTokenizer.from_pretrained(mid, trust_remote_code=False)
    if tk.pad_token is None: tk.pad_token = tk.eos_token
    tk.padding_side = "left"
    kw = dict(attn_implementation="sdpa", low_cpu_mem_usage=True, trust_remote_code=False)
    try:
        m = AutoModelForCausalLM.from_pretrained(mid, dtype=DTYPE, **kw)
    except TypeError:                          # transformers < 4.56
        m = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=DTYPE, **kw)
    # Ensure tied input/output embeddings remain mapped to shared matrix weights
    if getattr(m.config, "tie_word_embeddings", False):
        m.tie_weights()
        oe, ie = m.get_output_embeddings(), m.get_input_embeddings()
        if oe is not None and oe.weight is not ie.weight:
            oe.weight = ie.weight
        assert m.get_output_embeddings().weight is m.get_input_embeddings().weight, \
            f"{mid}: output layer is not tied to the input embeddings"
    assert not any(p.device.type == "meta" for p in m.parameters()), \
        f"{mid}: some weights were never loaded (meta tensors)"
    m = m.to(device=dev, dtype=DTYPE).eval()
    gc.collect()
    if dev.startswith("cuda"): torch.cuda.empty_cache()
    assert next(m.parameters()).device.type == ("cuda" if dev.startswith("cuda") else "cpu"),         f"{mid} on {next(m.parameters()).device}, expected {dev}"
    assert next(m.parameters()).dtype == DTYPE, f"{mid} loaded as {next(m.parameters()).dtype}"
    m.generation_config.pad_token_id = tk.pad_token_id
    _sanity_check(m, tk, mid)
    return m, tk

# ── per-thread model context ────────────────────────────────────────────────
# Each worker owns one GPU and one solver, and that solver is also its own critic.
# The paper already specifies self-verification, so this costs nothing
# methodologically and lets two GPUs generate at the same time instead of one
# generating while the other waits to verify.
class Ctx:
    def __init__(self, mid, dev):
        self.mid, self.dev = mid, dev
        self.solver, self.tok = load_lm(mid, dev)
        self.critic, self.ctok = self.solver, self.tok      # self-verification
        enc = lambda w: self.ctok.encode(w, add_special_tokens=False)[-1]
        self.YES = sorted({enc(w) for w in [" Yes", "Yes", " yes"]})
        self.NO  = sorted({enc(w) for w in [" No",  "No",  " no"]})
        _both = set(self.YES) & set(self.NO)        # never let a shared piece into both
        self.YES = [t for t in self.YES if t not in _both]
        self.NO  = [t for t in self.NO  if t not in _both]
        assert self.YES and self.NO, f"{mid}: could not find distinct Yes/No tokens"
    def free(self):
        self.solver = self.critic = None
        gc.collect()
        if self.dev.startswith("cuda"): torch.cuda.empty_cache()

_TL = threading.local()
def CTX():
    c = getattr(_TL, "ctx", None)
    if c is None:
        raise RuntimeError("no model context bound to this thread")
    return c

STATS_LOCK = threading.Lock()

class StreamEnt(LogitsProcessor):
    def __init__(self, n, T): self.buf = torch.zeros(n, T); self.t = 0
    def __call__(self, ids, sc):
        if self.t < self.buf.shape[1]:
            lp = torch.log_softmax(sc.float(), -1)
            self.buf[:sc.shape[0], self.t] = (-(lp.exp()*lp).sum(-1)).cpu(); self.t += 1
        return sc

FIN = re.compile(r"final answer\s*[::]", re.I)
BOX = re.compile(r"\\boxed\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}")

class DoneStop(StoppingCriteria):
    """Stop rows that have finished their answer.
    Decodes only the last `tail` tokens of rows still running, every `every` steps.
    Accepts closed \\boxed{...} or `Final Answer: ...` as finished."""
    def __init__(self, tk, start, n, every=16, tail=56):
        self.tk, self.s, self.e, self.tail = tk, start, every, tail
        self.done = torch.zeros(n, dtype=torch.bool); self.i = 0
        self.eos = tk.eos_token_id
    def __call__(self, ids, sc, **kw):
        self.i += 1
        if self.i % self.e:
            return self.done.to(ids.device)
        live = (~self.done).nonzero().flatten().tolist()
        if not live:
            return self.done.to(ids.device)
        gen_ids = ids[live, self.s:]
        if self.eos is not None:
            hit = (gen_ids == self.eos).any(-1).cpu()
            for r, h in zip(live, hit.tolist()):
                if h: self.done[r] = True
        txt = self.tk.batch_decode(gen_ids[:, -self.tail:], skip_special_tokens=True)
        for r, t in zip(live, txt):
            if self.done[r]: continue
            m = FIN.search(t)
            if (m and ("\n" in t[m.end():] or len(t[m.end():]) >= 24)) or BOX.search(t):
                self.done[r] = True
        return self.done.to(ids.device)

SMALL_BATCH = {
    "microsoft/Phi-3.5-mini-instruct": 6_000,
    "Qwen/Qwen2.5-3B-Instruct": 12_000,
    "Qwen/Qwen2.5-Math-1.5B-Instruct": 16_000,
    "HuggingFaceTB/SmolLM2-1.7B-Instruct": 16_000,
    "Qwen/Qwen2.5-1.5B-Instruct": 18_000
}
_ADJ = {}   # model id -> token budget lowered after an OOM; never raised again

def _model_key(model):
    return getattr(model.config, "_name_or_path", "") or str(id(model))

def _budget(model, default):
    """Token budget per batch for this model, adaptively lowered after OOMs."""
    k = _model_key(model)
    base = min(default, SMALL_BATCH.get(k, default))
    return min(base, _ADJ.get(k, base))

def _shrink_budget(model, floor=2_000):
    """After an OOM, permanently lower this model's batch budget for the rest
    of the run, so later batches stop re-hitting the same wall."""
    k = _model_key(model)
    cur = _ADJ.get(k)
    if cur is None:
        cur = min(CFG.batch_tok, SMALL_BATCH.get(k, CFG.batch_tok))
    _ADJ[k] = max(floor, int(cur*0.6))
    return _ADJ[k]

def pack(lens, n, new, budget, cap):
    order = sorted(range(len(lens)), key=lambda i: lens[i])
    out, cur, mx = [], [], 0
    for i in order:
        m = max(mx, lens[i])
        # Ensure total sequence count and memory budget are respected
        if cur and ((len(cur)+1)*n > cap or (len(cur)+1)*n*(m+new) > budget):
            out.append(cur); cur, mx = [i], lens[i]
        else: cur.append(i); mx = m
    if cur: out.append(cur)
    return out

@torch.inference_mode()
def _free(model):
    gc.collect()
    if model.device.type == "cuda": torch.cuda.empty_cache()

def _stitch(r1, k1, r2, k2, P):
    """Join two sample passes back into prompt-major order: for each prompt, its k1
    samples from the first pass followed by its k2 samples from the second."""
    (d1, n1, e1), (d2, n2, e2) = r1, r2
    dec, nt, E = [], [], ([] if (e1 is not None and e2 is not None) else None)
    for j in range(P):
        dec += list(d1[j*k1:(j+1)*k1]) + list(d2[j*k2:(j+1)*k2])
        nt  += list(n1[j*k1:(j+1)*k1]) + list(n2[j*k2:(j+1)*k2])
        if E is not None:
            E += list(e1[j*k1:(j+1)*k1]) + list(e2[j*k2:(j+1)*k2])
    return dec, nt, E

def _gen_single_batch(model, tok, ch, new, temp, n, greedy, ent, stop):
    """Generate n samples for each prompt in ch, splitting the samples on OOM.

    The retry deliberately happens AFTER leaving the except block. Inside `except`,
    the exception's traceback keeps the failed generate() frame alive, and with it the
    whole KV cache of the attempt that just failed, so memory cannot be released and
    the retry starts already full. That is what made every fallback fail before."""
    enc = tok(ch, return_tensors="pt", padding=True, truncation=True, max_length=1536).to(model.device)
    st = enc["input_ids"].shape[1]

    def _run_pass(sub_n):
        ns = len(ch) * sub_n
        lp = LogitsProcessorList()
        ep = StreamEnt(ns, new) if ent else None
        if ep: lp.append(ep)
        sc = StoppingCriteriaList([DoneStop(tok, st, ns)]) if stop else StoppingCriteriaList()
        o = model.generate(**enc, max_new_tokens=new, do_sample=not greedy,
                           temperature=max(temp, 1e-5), top_p=CFG.top_p if not greedy else None,
                           num_return_sequences=sub_n, logits_processor=lp, stopping_criteria=sc,
                           pad_token_id=tok.pad_token_id, use_cache=True)
        seq = o[:, st:]
        dec = tok.batch_decode(seq, skip_special_tokens=True)
        nt = (seq != tok.pad_token_id).sum(-1).tolist()
        E = ep.buf[:, :ep.t].numpy() if ep else None
        del o, seq
        return dec, nt, E

    def _attempt(k):
        try:
            return _run_pass(k)
        except OOMErr:
            pass                               # leave the except block first
        _free(model); _shrink_budget(model)    # the failed attempt is now unreachable
        if k == 1:
            raise OOMErr(f"out of memory on a single sequence ({model.device})")
        k1 = k // 2; k2 = k - k1
        print(f"    [{model.device}] OOM on n={k} -> freed memory, retrying as {k1} + {k2}")
        r1 = _attempt(k1)
        r2 = _attempt(k2)
        return _stitch(r1, k1, r2, k2, len(ch))

    try:
        return _attempt(n)
    finally:
        del enc

@torch.inference_mode()
def gen(model, tok, prompts, new, temp=0.7, n=1, ent=False, stop=True):
    if not prompts: return [], []
    greedy = temp <= 0; n = 1 if greedy else n
    lens = [len(x) for x in tok(prompts, add_special_tokens=False)["input_ids"]]
    T, M = [None]*len(prompts), [None]*len(prompts); t0 = time.time()
    queue = list(pack(lens, n, new, _budget(model, CFG.batch_tok), CFG.batch_seq))
    nb_total, nb_done, t_beat = len(queue), 0, time.time()
    while queue:
        idx = queue.pop(0)
        ch = [prompts[i] for i in idx]
        failed = False
        try:
            dec, nt, E = _gen_single_batch(model, tok, ch, new, temp, n, greedy, ent, stop)
        except OOMErr:
            failed = True                      # handled below, outside the except block
        if failed:
            _free(model); _shrink_budget(model)
            if len(idx) > 1:
                h = len(idx)//2
                queue = [idx[:h], idx[h:]] + queue
                nb_total += 1
                print(f"    [{model.device}] OOM -> splitting prompt batch of {len(idx)}")
                continue
            # one prompt, one sample at a time still failed: shorter generations
            print(f"    [{model.device}] OOM on single prompt -> retrying with shorter max_new")
            ok = False
            for red in range(1, 4):
                bad = False
                try:
                    dec, nt, E = _gen_single_batch(model, tok, ch, max(64, new >> red),
                                                   temp, n, greedy, ent, stop)
                    ok = True
                except OOMErr:
                    bad = True
                if ok: break
                if bad: _free(model)
            if not ok:
                raise OOMErr(f"generation failed after all fallbacks ({model.device})")

        for j, gi in enumerate(idx):
            sl = slice(j*n, (j+1)*n); T[gi] = dec[sl]
            M[gi] = [{"tok": int(nt[r]), "ent": (E[r] if E is not None else None)}
                     for r in range(sl.start, sl.stop)]
        with STATS_LOCK: STATS["tok"] += sum(nt)
        # release cached blocks every batch so fragmentation cannot build up over
        # Garbage collection and cache release during batched inference
        if model.device.type == "cuda": torch.cuda.empty_cache()
        nb_done += 1
        if time.time() - t_beat > 300:          # heartbeat every 5 min
            print(f"    [{model.device}] batch {nb_done}/{nb_total} | {el()}")
            t_beat = time.time()
    with STATS_LOCK: STATS["sec"] += time.time()-t0
    return T, M

SYS = ("You are a careful reasoner. Think step by step, keep it tight, then write "
       "`Final Answer: <answer>` on the last line.")
def chat(tok, u, s=SYS):
    try:
        return tok.apply_chat_template([{"role":"system","content":s},{"role":"user","content":u}],
                                       tokenize=False, add_generation_prompt=True)
    except Exception:   # some checkpoints have no system role
        return tok.apply_chat_template([{"role":"user","content":s+"\n\n"+u}],
                                       tokenize=False, add_generation_prompt=True)
def qp(it, style="cot"):
    tail = "\nPut the final expression in \\boxed{}." if it.kind=="latex" else "\nGive the final numeric answer."
    lead = {"cot":"Solve this problem.","plan":"First write a 2-line plan, then execute it.",
            "short":"Solve with the shortest correct reasoning."}[style]
    return f"{lead}\n\n{it.question}{tail}"
def thr(): return f"{STATS['tok']:,} tok / {STATS['sec']/60:.1f} min = {STATS['tok']/max(STATS['sec'],1):.0f} tok/s"

# ---------- verifier ----------
SYSV = "You are a strict grader. Reply with a single word: Yes or No."

def _last_logits(model, enc):
    """Compute lm_head on the LAST position only to save memory."""
    try:
        return model(**enc, logits_to_keep=1, use_cache=False).logits[:, -1, :].float()
    except (TypeError, ValueError):
        pass
    try:                                   # transformers < 4.45 spelling
        return model(**enc, num_logits_to_keep=1, use_cache=False).logits[:, -1, :].float()
    except (TypeError, ValueError):
        pass
    # Extract representation from final token hidden state
    h = model.model(**enc, use_cache=False).last_hidden_state[:, -1, :]
    return model.lm_head(h).float()

@torch.inference_mode()
def critic_yes(pairs):
    if not pairs: return []
    C = CTX(); CTOK, CRITIC = C.ctok, C.critic
    pr = [CTOK.apply_chat_template([{"role":"system","content":SYSV},
          {"role":"user","content":f"Problem:\n{q}\n\nSolution so far:\n{s[-1500:]}\n\n"
           "Is the reasoning correct so far and heading to the right answer?"}],
          tokenize=False, add_generation_prompt=True) for q, s in pairs]
    lens = [len(x) for x in CTOK(pr, add_special_tokens=False)["input_ids"]]
    out = [0.0]*len(pairs)
    queue = list(pack(lens, 1, 0, _budget(CRITIC, CFG.critic_tok), CFG.batch_seq))
    while queue:
        idx = queue.pop(0)
        try:
            enc = CTOK([pr[i] for i in idx], return_tensors="pt", padding=True,
                       truncation=True, max_length=2048).to(CRITIC.device)
            lg = _last_logits(CRITIC, enc)
            y = torch.logsumexp(lg[:, C.YES], -1); n = torch.logsumexp(lg[:, C.NO], -1)
            for i, v in zip(idx, torch.sigmoid(y-n).cpu().tolist()): out[i] = v
            del enc, lg
            if CRITIC.device.type == "cuda": torch.cuda.empty_cache()
        except OOMErr:
            if CRITIC.device.type == "cuda": torch.cuda.empty_cache()
            if len(idx) == 1:
                out[idx[0]] = 0.5; continue        # neutral score fallback
            h = len(idx)//2
            queue = [idx[:h], idx[h:]] + queue
            CFG.critic_tok = max(4_000, int(CFG.critic_tok*0.6))
            print(f"    [critic OOM] splitting batch, critic_tok -> {CFG.critic_tok:,}")
    return out

# ---------- topologies ----------
def run_sc(items, k):
    C = CTX(); SOLVER, STOK = C.solver, C.tok
    T, M = gen(SOLVER, STOK, [chat(STOK, qp(it)) for it in items], CFG.max_new, CFG.temp, n=k)
    out = []
    for it, tx, mt in zip(items, T, M):
        pr = [extract(t, it.kind) for t in tx]
        out.append({"item": it, "texts": tx, "preds": pr, "tok": [m["tok"] for m in mt],
                    "ok": [grade(p, it.gold, it.kind) for p in pr]})
    return out

def run_tree(items, n_init=1, beam=None, depth=None):
    if not items: return []
    C = CTX(); SOLVER, STOK = C.solver, C.tok
    beam = beam or CFG.tot_beam; depth = depth or CFG.tot_depth
    seg = max(140, int(CFG.max_new*0.55)//max(1, depth-1))
    states = [[""]*n_init for _ in items]; tk = [0]*len(items)
    for d in range(depth):
        last = (d == depth-1)
        pr, own, par = [], [], []
        for i, (it, ss) in enumerate(zip(items, states)):
            base = chat(STOK, qp(it, "plan" if d == 0 else "cot"))
            for sX in ss: pr.append(base+sX); own.append(i); par.append(sX)
        nnew = (CFG.max_new - min(seg*(depth-1), CFG.max_new-120)) if last else seg
        T, M = gen(SOLVER, STOK, pr, max(96, nnew), CFG.temp, n=beam, stop=last)
        cand = defaultdict(list)
        for pi, (tx, mt) in enumerate(zip(T, M)):
            for t, m in zip(tx, mt): cand[own[pi]].append(par[pi]+t); tk[own[pi]] += m["tok"]
        if last: states = [cand[i] for i in range(len(items))]; break
        flat, span = [], []
        for i, it in enumerate(items):
            span.append((len(flat), len(flat)+len(cand[i]))); flat += [(it.question, c) for c in cand[i]]
        sc = critic_yes(flat)
        for i, (a, b) in enumerate(span):
            v = np.asarray(sc[a:b]); keep = np.argsort(-v)[:max(beam, n_init)]
            states[i] = [cand[i][j] for j in keep] or [""]
    out = []
    for i, it in enumerate(items):
        pr_ = [extract(t, it.kind) for t in states[i]]
        mv, _ = majority(pr_, it.kind)
        if mv is None and states[i]:
            v = critic_yes([(it.question, t) for t in states[i]])
            mv = extract(states[i][int(np.argmax(v))], it.kind)
        out.append({"item": it, "texts": states[i], "tok": tk[i], "ok": grade(mv, it.gold, it.kind)})
    return out

ACTIONS = ([("SC", b) for b in CFG.budgets] + [("BoN", b) for b in CFG.budgets if b > 1]
           + [("ToT", 4), ("ToT", 8), ("FoT", 8)])
ACT_1D = [("SC", b) for b in CFG.budgets]
print(f"action space |A|={len(ACTIONS)} | 1-D (Zhai) |A|={len(ACT_1D)}")


action space |A|=12 | 1-D (Zhai) |A|=5


## 3. Empirical Utility Tensor Construction & Checkpoint Pipeline

Executes per-model generation across all 200 benchmark questions, building empirical accuracy $\mathrm{Acc}(q, a)$ and cost $\mathrm{Cost}(q, a)$ utility tensors.


In [4]:
MSTEP = re.compile(r"\b(then|after|next|first|second|finally|total|altogether|remaining)\b", re.I)
def base16(it):
    q = it.question; w = q.split()
    nums = [float(x) for x in re.findall(r"-?\d+\.?\d*", q.replace(",", ""))] or [0.0]
    return {"len_chars":len(q), "len_words":len(w), "n_sent":q.count(".")+q.count("?"),
      "n_qmark":q.count("?"), "n_numbers":len(nums),
      "num_mag_avg":math.log1p(abs(np.mean(nums))), "num_mag_max":math.log1p(abs(np.max(np.abs(nums)))),
      "has_pct":int("%" in q or "percent" in q.lower()), "has_frac":int("/" in q or "\\frac" in q),
      "has_time":int(bool(re.search(r"\b(hour|minute|day|week|year|time)\b", q, re.I))),
      "has_money":int(bool(re.search(r"\$|dollar|cost|price", q, re.I))),
      "has_rate":int(bool(re.search(r"\b(rate|speed|per)\b", q, re.I))),
      "has_multistep":len(MSTEP.findall(q)),
      "avg_word_len":float(np.mean([len(x) for x in w])) if w else 0.0,
      "uniq_ratio":len(set(w))/max(1, len(w)),
      "has_sqrt":int("sqrt" in q.lower() or "\\sqrt" in q)}
BASE_KEYS = sorted(base16(ITEMS[0]).keys())

def semantic_entropy(preds, kind, kp=None):
    """Normalized semantic entropy over equivalence classes, Eq. (6) in paper."""
    kp = kp or CFG.probe_k
    v = [p for p in preds if p is not None]
    if not v: return 1.0
    c = np.array(list(Counter(kf(kind)(x) for x in v).values()), float); p = c/c.sum()
    norm = math.log(max(2, kp))
    return float(np.clip(-(p*np.log(p+1e-12)).sum() / norm, 0.0, 1.0))

def sc_curve(res, budgets, n_sample):
    it = res["item"]; pr, ok = res["preds"], res["ok"]; k = kf(it.kind); out = {}
    for b in budgets:
        w = n_sample // b
        if w == 0: out[b] = out.get(max([x for x in budgets if x < b], default=b), 0.0); continue
        hits = []
        for j in range(w):
            sub = slice(j*b, (j+1)*b)
            mv, _ = majority(pr[sub], it.kind)
            hits.append(int(mv is not None and any(ok[i] and pr[i] is not None and k(pr[i])==k(mv)
                                                   for i in range(sub.start, sub.stop))))
        out[b] = float(np.mean(hits))
    return out

def _atomic_pickle(obj, path):
    """Write to a temp file and rename, so a kill mid-write cannot corrupt a checkpoint."""
    with open(path + ".tmp", "wb") as f: pickle.dump(obj, f)
    os.replace(path + ".tmp", path)
    sync_up(path)

def build_model(mid, dev):
    """Everything for one model, on one GPU, checkpointed after every stage.

    Five stages: the SC pool, three tree topologies, and the probe. Each is pickled
    the moment it finishes. The pool is additionally checkpointed per 25-question
    chunk and each tree topology per run, so an interrupted session loses minutes,
    not hours. On restart, finished stages are loaded and the model is only put
    on the GPU if some stage still has to run."""
    short = mid.split("/")[-1]; t0 = time.time()

    def need():
        if getattr(_TL, "ctx", None) is None:
            _TL.ctx = Ctx(mid, dev)
            mem = (f"{torch.cuda.memory_allocated(int(dev[-1]))/1e9:.1f} GB"
                   if dev.startswith("cuda") else "cpu")
            print(f"  [{dev}] {short} loaded, {mem}")
        return _TL.ctx

    def stage(name, fn):
        p = f"{CFG.out}/ckpt/{short}_{len(ITEMS)}q_{name}.pkl"
        if CFG.resume and os.path.exists(p):
            with open(p, "rb") as f: v = pickle.load(f)
            print(f"  [{dev}] {short}: stage '{name}' restored")
            return v
        need(); ts = time.time()
        v = fn()
        _atomic_pickle(v, p)
        print(f"  [{dev}] {short}: stage '{name}' done in {(time.time()-ts)/60:.1f} min [{el()}]")
        return v

    # Stage 1: Self-consistency pool generation and verifier scoring (checkpointed in 25-item chunks)
    def _pool():
        part_p = f"{CFG.out}/ckpt/{short}_{len(ITEMS)}q_pool_parts.pkl"
        parts = {}
        if CFG.resume and os.path.exists(part_p):
            with open(part_p, "rb") as f: parts = pickle.load(f)
            print(f"  [{dev}] {short}: pool resuming at {len(parts)}/{len(ITEMS)} questions")
        CH = 25
        for i0 in range(0, len(ITEMS), CH):
            chunk = [it for it in ITEMS[i0:i0+CH] if it.qid not in parts]
            if not chunk: continue
            rs = run_sc(chunk, CFG.n_sample)
            flat, span = [], []
            for r in rs:
                span.append((len(flat), len(flat)+len(r["texts"])))
                flat += [(r["item"].question, t) for t in r["texts"]]
            V = critic_yes(flat)
            for r, (a, b) in zip(rs, span):
                parts[r["item"].qid] = {"preds": r["preds"], "ok": r["ok"],
                                        "tok": r["tok"], "v": list(V[a:b])}
            _atomic_pickle(parts, part_p)
            print(f"  [{dev}] {short}: pool {len(parts)}/{len(ITEMS)} questions | {el()}")
        slim_pool = [{"qid": it.qid, "preds": parts[it.qid]["preds"],
                      "ok": parts[it.qid]["ok"], "tok": parts[it.qid]["tok"]}
                     for it in ITEMS]
        vt = {it.qid: parts[it.qid]["v"] for it in ITEMS}
        return slim_pool, vt
    POOL, VTRACE = stage("pool", _pool)
    PIDX = {r["qid"]: r for r in POOL}

    ACC, COST, QCONF = {}, {}, {}
    for it in ITEMS:
        q = it.qid; r = PIDX[q]
        c = sc_curve({"item": it, "preds": r["preds"], "ok": r["ok"]}, CFG.budgets, CFG.n_sample)
        mt = float(np.mean(r["tok"])); v = VTRACE[q]
        for b in CFG.budgets:
            ACC[(q,"SC",b)] = c[b]; COST[(q,"SC",b)] = mt*b
            QCONF[(q,"SC",b)] = float(np.mean(v[:b]))
            if b == 1: continue
            w = max(1, CFG.n_sample//b); hits = []
            for jj in range(w):
                sl = slice(jj*b, (jj+1)*b)
                if len(v[sl]) == 0: continue
                hits.append(int(r["ok"][sl.start + int(np.argmax(v[sl]))]))
            ACC[(q,"BoN",b)] = float(np.mean(hits)) if hits else 0.0
            COST[(q,"BoN",b)] = mt*b*1.05
            QCONF[(q,"BoN",b)] = float(np.max(v[:b]))

    # Stages 2-4: Tree search topologies (ToT/FoT) evaluated over stochastic runs
    for name, ninit, bud, bm in [("ToT",1,4,2), ("ToT",1,8,CFG.tot_beam),
                                 ("FoT",CFG.fot_trees,8,CFG.tot_beam)]:
        def _tree(ninit=ninit, bm=bm, sname=name, sbud=bud):
            n_runs = max(1, getattr(CFG, "tree_runs", 1))
            run_p = f"{CFG.out}/ckpt/{short}_{len(ITEMS)}q_{sname}{sbud}_runs.pkl"
            run_results = []
            if CFG.resume and os.path.exists(run_p):
                with open(run_p, "rb") as f: run_results = pickle.load(f)
                print(f"  [{dev}] {short}: {sname}@{sbud} resuming at run {len(run_results)}/{n_runs}")
            while len(run_results) < n_runs:
                rs = run_tree(ITEMS, n_init=ninit, beam=bm)
                flat, span = [], []
                for r in rs:
                    span.append((len(flat), len(flat)+len(r["texts"])))
                    flat += [(r["item"].question, t) for t in r["texts"]]
                V = critic_yes(flat) if flat else []
                run_results.append([{"qid": r["item"].qid, "ok": float(r["ok"]), "tok": float(r["tok"]),
                                     "conf": float(np.mean(V[a:b])) if b > a else 0.0}
                                    for r, (a, b) in zip(rs, span)])
                _atomic_pickle(run_results, run_p)
                print(f"  [{dev}] {short}: {sname}@{sbud} run {len(run_results)}/{n_runs} done | {el()}")
            combined = []
            for idx in range(len(ITEMS)):
                qid = run_results[0][idx]["qid"]
                mean_ok = float(np.mean([run_results[ri][idx]["ok"] for ri in range(n_runs)]))
                mean_tok = float(np.mean([run_results[ri][idx]["tok"] for ri in range(n_runs)]))
                mean_conf = float(np.mean([run_results[ri][idx]["conf"] for ri in range(n_runs)]))
                combined.append({"qid": qid, "ok": mean_ok, "tok": mean_tok, "conf": mean_conf})
            return combined

        res = stage(f"{name}{bud}", _tree)
        for r in res:
            ACC[(r["qid"],name,bud)] = r["ok"]; COST[(r["qid"],name,bud)] = r["tok"]
            QCONF[(r["qid"],name,bud)] = r["conf"]
        print(f"  [{dev}] {short}: {name}@{bud} acc={np.mean([r['ok'] for r in res]):.3f}")

    unit = float(np.mean([COST[(q,"SC",1)] for q in QIDS]))
    CN = {k: v/unit for k, v in COST.items()}

    # Stage 5: Probe features, token entropy, and semantic clustering entropy
    def _probe():
        C = CTX(); SOLVER, STOK = C.solver, C.tok
        PT, PM = gen(SOLVER, STOK, [chat(STOK, qp(it,"short")) for it in ITEMS],
                     CFG.probe_tok, CFG.temp, n=CFG.probe_k, ent=True, stop=False)
        tails, owner, PR = [], [], {}
        for it, tx, mt in zip(ITEMS, PT, PM):
            e = [m["ent"] for m in mt if m["ent"] is not None and len(m["ent"]) > 4]
            PR[it.qid] = {"texts": tx, "tok": sum(m["tok"] for m in mt),
                          "tok_ent": float(np.mean(np.concatenate(e))) if e else 1.0}
            for t in tx: tails.append(chat(STOK, qp(it))+t); owner.append(it)
        TT, TM = gen(SOLVER, STOK, tails, CFG.max_new-CFG.probe_tok, CFG.temp, n=1)
        agg = defaultdict(list)
        for it, tx, mt in zip(owner, TT, TM): agg[it.qid].append((tx[0], mt[0]["tok"]))
        pflat, pspan = [], []
        for it in ITEMS:
            full = [a+b for a, b in zip(PR[it.qid]["texts"], [c[0] for c in agg[it.qid]])]
            PR[it.qid]["full"] = full
            pspan.append((len(pflat), len(pflat)+len(full)))
            pflat += [(it.question, t) for t in full]
        PV = critic_yes(pflat)
        for it, (pa, pb) in zip(ITEMS, pspan):
            d = PR[it.qid]; v = PV[pa:pb]
            d["cont_tok"] = sum(c[1] for c in agg[it.qid])
            pr = [extract(t, it.kind) for t in d["full"]]
            d["sem_ent"] = semantic_entropy(pr, it.kind)
            d["v_var"] = float(np.var(v)) if len(v) > 1 else 0.0        # SC vs BoN
            mv, agree = majority(pr, it.kind)
            d["p_agree"] = float(agree)                                  # budget
            k_ = kf(it.kind); best = pr[int(np.argmax(v))] if len(v) else None
            d["rerank_flip"] = float(best is not None and mv is not None
                                     and k_(best) != k_(mv))             # BoN
            fs = [next((l.strip()[:80] for l in t.strip().split("\n") if len(l.strip()) > 12),
                       t.strip()[:80]) for t in d["texts"]]
            d["first_div"] = len(set(fs)) / max(1, len(fs))             # trees
            d.pop("texts"); d.pop("full")                                # keep ckpt small
        return PR
    PROBE = stage("probe", _probe)

    Cb = {a: float(np.mean([CN[(q,)+a] for q in QIDS])) for a in ACTIONS}
    Ab = {a: float(np.mean([ACC[(q,)+a] for q in QIDS])) for a in ACTIONS}
    # Pareto dominance filter: identifies non-dominated actions in (accuracy, token cost) space
    DOM = {a: [b for b in ACTIONS if b != a and
               Cb[b] <= Cb[a] + 1e-9 and Ab[b] >= Ab[a] - 1e-9 and
               (Cb[b] < Cb[a] - 1e-9 or Ab[b] > Ab[a] + 1e-9)]
           for a in ACTIONS}
    PAR = [a for a in ACTIONS if not DOM[a]]
    print(f"  [{dev}] {short}: Pareto {PAR}")
    out = {"mid": mid, "ACC": ACC, "CN": CN, "QCONF": QCONF, "VTRACE": VTRACE, "POOL": POOL,
           "PROBE": PROBE, "Cb": Cb, "Ab": Ab, "PARETO": PAR, "unit": unit,
           "probe_saving": float(np.mean([PROBE[q]["tok"] for q in QIDS]))/unit,
           "minutes": (time.time()-t0)/60}
    if getattr(_TL, "ctx", None) is not None:
        _TL.ctx.free(); _TL.ctx = None
    return out

import pickle

def ck_path(mid):
    return f"{CFG.out}/ckpt/{mid.split('/')[-1]}_{len(ITEMS)}q.pkl"

def slim(md):
    """POOL and PROBE are already stripped of raw text inside build_model."""
    return md

MD = {}
_MD_LOCK = threading.Lock()

def _one_model(mid, dev):
    """Build one model on one GPU, checkpoint it, refresh costs.csv."""
    p = ck_path(mid)
    if CFG.resume and os.path.exists(p):
        with open(p, "rb") as f: md = pickle.load(f)
        print(f"[{dev}] {mid.split('/')[-1]}: restored from checkpoint "
              f"({os.path.getsize(p)/1e6:.0f} MB) | Pareto: {md['PARETO']}")
    else:
        print(f"[{dev}] building {mid.split('/')[-1]} ...")
        md = slim(build_model(mid, dev))
        with open(p, "wb") as f: pickle.dump(md, f)
        sync_up(p)
        print(f"[{dev}] {mid.split('/')[-1]} done in {md['minutes']:.1f} min "
              f"-> {os.path.basename(p)}")
    with _MD_LOCK:
        MD[mid] = md
        rows = []
        for m2, md2 in MD.items():
            par = set(md2["PARETO"])
            for a in md2["Cb"]:
                rows.append({"model": m2.split("/")[-1], "action": f"{a[0]}@{a[1]}",
                             "topology": a[0], "budget": a[1],
                             "cost": float(md2["Cb"][a]), "acc": float(md2["Ab"][a]),
                             "pareto": a in par})
        pd.DataFrame(rows).sort_values(["model", "cost"]).to_csv(
            f"{CFG.out}/costs.csv", index=False)
    return mid

# Concurrent multi-model execution across available GPUs
DEVS = [f"cuda:{i}" for i in range(N_GPU)] or ["cpu"]
n_work = len(DEVS) if CFG.parallel else 1
print(f"dispatching {len(MODELS)} models over {n_work} worker(s): {DEVS[:n_work]}")

t_build = time.time()
if n_work == 1:
    for mid in MODELS: _one_model(mid, DEVS[0])
else:
    from queue import Queue
    work = Queue()
    for mid in MODELS: work.put(mid)

    FAILED = []
    def worker(dev):
        if dev.startswith("cuda"): torch.cuda.set_device(int(dev.split(":")[1]))
        while True:
            try: mid = work.get_nowait()
            except Exception: return
            try:
                _one_model(mid, dev)
            except Exception as e:
                import traceback
                print(f"[{dev}] {mid.split('/')[-1]} FAILED: {type(e).__name__}: {e}")
                traceback.print_exc()
                FAILED.append(mid)
                if getattr(_TL, "ctx", None) is not None:
                    _TL.ctx.free(); _TL.ctx = None
            finally:
                work.task_done()

    with ThreadPoolExecutor(max_workers=n_work) as ex:
        for f in [ex.submit(worker, d) for d in DEVS[:n_work]]: f.result()

    for mid in FAILED:
        print(f"retrying {mid.split('/')[-1]} sequentially on {DEVS[0]}")
        _one_model(mid, DEVS[0])

missing = [m for m in MODELS if m not in MD]
assert not missing, f"these models did not finish: {missing}"
MD = {m: MD[m] for m in MODELS}            # restore deterministic order
print(f"\nall {len(MD)} models built in {(time.time()-t_build)/60:.1f} min wall clock | {thr()}")


dispatching 2 models over 1 worker(s): ['cuda:0']
[cuda:0] Qwen2.5-3B-Instruct: restored from checkpoint (0 MB) | Pareto: [('SC', 1), ('SC', 2), ('SC', 4), ('SC', 16), ('BoN', 2), ('ToT', 4), ('ToT', 8)]
[cuda:0] building Phi-3.5-mini-instruct ...
  [cuda:0] Phi-3.5-mini-instruct: stage 'pool' restored
  [cuda:0] Phi-3.5-mini-instruct: stage 'ToT4' restored
  [cuda:0] Phi-3.5-mini-instruct: ToT@4 acc=0.555


config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

Validating:           |   0%            

  [backup] saved Phi-3.5-mini-instruct_200q_FoT8.pkl
  [backup] saved Phi-3.5-mini-instruct_200q_FoT8_runs.pkl
    [cuda:0] batch 14/74 | 286.3 min
    [cuda:0] batch 29/74 | 291.6 min
    [cuda:0] batch 44/74 | 296.7 min
    [cuda:0] batch 57/74 | 301.8 min
    [cuda:0] batch 70/74 | 307.0 min
  [cuda:0] Phi-3.5-mini-instruct: stage 'probe' done in 32.6 min [310.7 min]
  [cuda:0] Phi-3.5-mini-instruct: Pareto [('SC', 1), ('SC', 2), ('SC', 4), ('SC', 8), ('SC', 16), ('BoN', 2), ('ToT', 4)]


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[cuda:0] Phi-3.5-mini-instruct done in 309.5 min -> Phi-3.5-mini-instruct_200q.pkl

all 2 models built in 309.5 min wall clock | 2,008,163 tok / 254.4 min = 132 tok/s


## 4. Optimization Framework: Lagrangian Dual & Surface Regression Allocator

Solves the constrained test-time allocation problem:
$$\max_{\{a_q\}} \sum_{q=1}^N \mathrm{Acc}(q, a_q) \quad \text{subject to} \quad \frac{1}{N} \sum_{q=1}^N \mathrm{Cost}(q, a_q) \le B$$

- **Pareto Filtering:** Discards strictly dominated actions.
- **Oracle Solvers:** Exact 1-D (budget-only) and 2-D (joint) Lagrangian dual solutions.
- **Learners S0–S5:** Compares discrete classification baselines against the REASON-X utility-surface regressor using 5-fold grouped cross-validation.


In [5]:
from sklearn.model_selection import GroupKFold, StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
try:
    from xgboost import XGBClassifier, XGBRegressor
    def mk_clf(sd): return XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                         verbosity=0, random_state=sd, n_jobs=2)
    def mk_reg(sd): return XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.08,
                                        subsample=0.9, verbosity=0, random_state=sd, n_jobs=2)
except Exception:
    from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
    def mk_clf(sd): return GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=sd)
    def mk_reg(sd): return GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=sd)

# Feature definitions: sequence length, arithmetic density, verifier score, and semantic entropy
PROBE_KEYS = ["v_var", "p_agree", "rerank_flip", "first_div"]

def featurize(md, use_sem, use_probe=None):
    if use_probe is None: use_probe = False
    rows = []
    for it in ITEMS:
        d = base16(it); d["tok_entropy"] = md["PROBE"][it.qid]["tok_ent"]
        if use_sem: d["sem_entropy"] = md["PROBE"][it.qid]["sem_ent"]
        if use_probe:
            for k_ in PROBE_KEYS: d[k_] = float(md["PROBE"][it.qid].get(k_, 0.0))
        rows.append(d)
    cols = (BASE_KEYS + ["tok_entropy"]
            + (["sem_entropy"] if use_sem else [])
            + (PROBE_KEYS if use_probe else []))
    return pd.DataFrame(rows)[cols].values.astype(np.float32)

def oracle_at(md, acts, lam):
    """Oracle action per question with Proposition 1 deterministic tie-breaking (prefer cheaper action)."""
    ACC, CN = md["ACC"], md["CN"]; out = {}
    for q in QIDS:
        best, bv = None, -1e18
        for a in acts:
            v = ACC[(q,)+a] - lam*CN[(q,)+a]
            # Deterministic tie-breaking favoring minimum computational cost (Proposition 1)
            if best is None or v > bv + 1e-9 or (abs(v - bv) <= 1e-9 and CN[(q,)+a] < CN[(q,)+best]):
                best, bv = a, v
        out[q] = best
    return out

def cbar_at(md, acts, lam):
    o = oracle_at(md, acts, lam)
    return float(np.mean([md["CN"][(q,)+o[q]] for q in QIDS]))

def check_monotonicity(md, acts, lo=0.0, hi=5.0, steps=25):
    """Numerical verification of Proposition 1 cost monotonicity on a grid."""
    grid = np.linspace(lo, hi, steps)
    costs = [cbar_at(md, acts, lam) for lam in grid]
    is_mono = all(costs[i] >= costs[i+1] - 1e-6 for i in range(len(costs)-1))
    return is_mono, costs

def find_lambda(md, acts, B, lo=0.0, hi=10.0, iters=50):
    for _ in range(iters):
        mid_ = (lo+hi)/2
        if cbar_at(md, acts, mid_) > B: lo = mid_
        else: hi = mid_
    return hi

def eval_policy(md, chosen):
    return (float(np.mean([md["ACC"][(q,)+chosen[q]] for q in QIDS])),
            float(np.mean([md["CN"][(q,)+chosen[q]] for q in QIDS])))

def alloc_from_surface(md, pred_acc, acts, B):
    """Same Lagrangian, run on a PREDICTED utility surface with true costs and Proposition 1 tie-breaking."""
    CN = md["CN"]
    def pick(lam):
        out = {}
        for q in QIDS:
            best, bv = None, -1e18
            for a in acts:
                v = pred_acc[(q,)+a] - lam*CN[(q,)+a]
                if best is None or v > bv + 1e-9 or (abs(v - bv) <= 1e-9 and CN[(q,)+a] < CN[(q,)+best]):
                    best, bv = a, v
            out[q] = best
        return out
    lo, hi = 0.0, 10.0
    for _ in range(50):
        m_ = (lo+hi)/2
        c = float(np.mean([CN[(q,)+pick(m_)[q]] for q in QIDS]))
        if c > B: lo = m_
        else: hi = m_
    return pick(hi)

def learner_classification(md, acts, B, use_sem, oracle_lab, seeds=None):
    seeds = seeds or list(range(CFG.n_seeds))
    X = featurize(md, use_sem)
    space = list(acts)
    lab = {a: i for i, a in enumerate(space)}
    y = np.array([lab[oracle_lab[q]] for q in QIDS])
    accs, costs, imits = [], [], []
    chosen = None
    for sd in seeds:
        pred = np.empty(len(QIDS), int)
        cnt = Counter(y); k_ok = min(5, min(cnt.values())) if len(cnt) > 1 else 1
        splits = (list(StratifiedKFold(k_ok, shuffle=True, random_state=sd).split(X, y))
                  if k_ok >= 2 else
                  list(KFold(min(5, max(2, len(QIDS)//3)), shuffle=True, random_state=sd).split(X)))
        for tr, te in splits:
            u = np.unique(y[tr])
            if len(u) < 2: pred[te] = u[0]; continue
            rm = {c: i for i, c in enumerate(u)}; inv = {i: c for c, i in rm.items()}
            m = mk_clf(sd); m.fit(X[tr], np.array([rm[v] for v in y[tr]]))
            pred[te] = np.array([inv[int(p)] for p in m.predict(X[te])])
        chosen = {q: space[int(p)] for q, p in zip(QIDS, pred)}
        a, c = eval_policy(md, chosen); accs.append(a); costs.append(c)
        imits.append(float(np.mean(pred == y)))
    return float(np.mean(accs)), float(np.std(accs)), float(np.mean(costs)), float(np.mean(imits)), chosen

def learner_surface(md, acts, B, use_sem, oracle_lab, seeds=None, use_probe=False):
    """Predict Acc(x,a) for every action, then allocate with the same Lagrangian."""
    seeds = seeds or list(range(CFG.n_seeds))
    X = featurize(md, use_sem, use_probe); space = list(acts)
    qi = {q: i for i, q in enumerate(QIDS)}
    rowsX, rowsY, groups, keys = [], [], [], []
    for q in QIDS:
        for j, a in enumerate(space):
            oh = np.zeros(len(space), np.float32); oh[j] = 1.0
            rowsX.append(np.concatenate([X[qi[q]], oh,
                          [math.log(a[1]), md["CN"][(q,)+a]]]).astype(np.float32))
            rowsY.append(md["ACC"][(q,)+a]); groups.append(qi[q]); keys.append((q,)+a)
    RX = np.stack(rowsX); RY = np.asarray(rowsY, np.float32); G = np.asarray(groups)
    accs, costs, imits, r2s = [], [], [], []
    chosen = None; pa_last = {}
    ng = min(5, max(2, len(QIDS)//3))
    uniq_g = np.unique(G)

    for sd in seeds:
        pa = {}
        # Grouped K-Fold cross-validation partitioned strictly by question ID to prevent data leakage
        rng_sd = np.random.default_rng(sd)
        perm_map = {g: new_g for new_g, g in enumerate(rng_sd.permutation(uniq_g))}
        G_shuffled = np.array([perm_map[g] for g in G])
        for tr, te in GroupKFold(n_splits=ng).split(RX, RY, groups=G_shuffled):
            m = mk_reg(sd); m.fit(RX[tr], RY[tr]); p = m.predict(RX[te])
            for i, v in zip(te, p): pa[keys[i]] = float(v)
        ss = float(np.sum((RY - RY.mean())**2))
        r2s.append(1.0 - float(np.sum([(pa[k]-md["ACC"][k])**2 for k in keys]))/max(ss, 1e-9))
        chosen = alloc_from_surface(md, pa, space, B); pa_last = pa
        a, c = eval_policy(md, chosen); accs.append(a); costs.append(c)
        imits.append(float(np.mean([chosen[q] == oracle_lab[q] for q in QIDS])))
    return (float(np.mean(accs)), float(np.std(accs)), float(np.mean(costs)),
            float(np.mean(imits)), chosen, float(np.mean(r2s)), pa_last)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

## 5. Comparative Evaluation Ladder (Stages S0–S5b)

Evaluates allocation policies across target budgets $B \in \{1.5, 2.0, 3.0\}$ using 5-fold grouped cross-validation across random seeds.


In [ ]:
ROWS = []
def rec(model, stage, method, B, acc, sd, cost, **kw):
    d = {"Model": model.split("/")[-1], "Stage": stage, "Method": method, "B": B,
         "Acc": acc, "Std": sd, "Cost": cost}; d.update(kw); ROWS.append(d)
    extra = "  ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in kw.items())
    print(f"  {stage:<4} {method:<32} B={B}  acc={acc:.4f}±{sd:.3f}  cost={cost:.2f}  {extra}")

UP = {("SC",1):("SC",2), ("SC",2):("SC",4), ("SC",4):("SC",8), ("SC",8):("SC",16),
      ("BoN",2):("BoN",4), ("BoN",4):("BoN",8), ("BoN",8):("BoN",16),
      ("ToT",4):("ToT",8), ("ToT",8):("FoT",8)}

def escalate(md, base_choice, acts, B, tau, ahat):
    ch = dict(base_choice); CN, QC = md["CN"], md["QCONF"]
    spent = float(np.mean([CN[(q,)+ch[q]] for q in QIDS])); pot = B - spent
    if pot <= 0: return ch, 0
    cand = []
    for q in QIDS:
        a = ch[q]; nxt = UP.get(a)
        if nxt is None or nxt == a or nxt not in acts: continue
        dc = CN[(q,)+nxt] - CN[(q,)+a]
        if dc <= 0: continue
        conf = QC[(q,)+a]
        gain = ahat.get((q,)+nxt, 0.0) - ahat.get((q,)+a, 0.0)
        cand.append((conf >= tau, conf, -gain/dc, q, nxt, dc))
    cand.sort()
    used, n = 0.0, 0
    for _, _, _, q, nxt, dc in cand:
        if used + dc/len(QIDS) > pot: continue
        ch[q] = nxt; used += dc/len(QIDS); n += 1
    return ch, n

CRC_GRID = np.round(np.linspace(0.95, 0.30, 24), 3)[::-1]; CRC_NMIN = 20
try:
    from scipy.stats import beta as _B; HAVE_SP = True
except Exception: HAVE_SP = False

def risk_ub(rhat, n, d):
    if n == 0: return 1.0
    if not HAVE_SP: return min(1.0, rhat + math.sqrt(math.log(1/d)/(2*n)))
    k = int(round(rhat*n))
    return 1.0 if k >= n else float(min(1.0, _B.ppf(1-d, k+1, n-k)))

def calibrate_crc(scores, ok, alpha, delta):
    s, y = np.asarray(scores), np.asarray(ok).astype(int)
    d = delta/len(CRC_GRID); rows = []
    for tau in CRC_GRID:
        m = s >= tau; n = int(m.sum()); r = float(1-y[m].mean()) if n else 0.0
        rows.append({"tau":float(tau), "n":n, "risk":r, "ub":risk_ub(r,n,d), "testable":n>=CRC_NMIN})
    tr = pd.DataFrame(rows); ok_ = tr[tr["testable"]]
    if len(ok_) == 0: return (float(np.quantile(s,0.9)) if len(s) else 1.0), 1.0, tr
    hit = ok_[ok_["ub"] <= alpha]
    row = hit.loc[hit["tau"].idxmin()] if len(hit) else ok_.loc[ok_["ub"].idxmin()]
    return float(row["tau"]), float(row["ub"]), tr

SUMMARY = {}
for mid in MODELS:
    md = MD[mid]; short = mid.split("/")[-1]
    print(f"\n{'='*76}\n{short}\n{'='*76}")
    acts2 = md["PARETO"] if len(md["PARETO"]) >= 3 else ACTIONS
    print(f"  prediction space restricted to Pareto set: {len(acts2)}/{len(ACTIONS)} actions")

    # Numerical verification of cost monotonicity across target budgets (Proposition 1)
    mono_1d, _ = check_monotonicity(md, ACT_1D)
    mono_2d, _ = check_monotonicity(md, acts2)
    print(f"  cost-monotonicity check (Proposition 1): 1-D={mono_1d}, 2-D={mono_2d}")

    # Conformal Risk Control (CRC) calibration for escalation threshold tau (Section 7.6)
    cs, co = [], []
    for r in md["POOL"][:max(2, min(100, len(QIDS)//2 if len(QIDS)>=200 else len(QIDS)))]:
        cs += list(md["VTRACE"][r["qid"]]); co += list(r["ok"])
    TAU, AACH, CRCTR = calibrate_crc(cs, co, CFG.alpha, CFG.delta)
    try: AUROC = float(roc_auc_score(co, cs))
    except Exception: AUROC = float("nan")
    print(f"  CRC n={len(cs)} tau={TAU:.3f} alpha_ach={AACH:.3f} AUROC={AUROC:.3f}")

    O1 = {}; O2 = {}; HR = {}
    for B in CFG.targets:
        O1[B] = oracle_at(md, ACT_1D, find_lambda(md, ACT_1D, B))
        O2[B] = oracle_at(md, acts2,  find_lambda(md, acts2,  B))
        HR[B] = eval_policy(md, O2[B])[0] - eval_policy(md, O1[B])[0]

    for B in CFG.targets:
        print(f"\n--- B = {B} (oracle headroom {100*HR[B]:+.2f} pp) ---")
        fb = max([b for b in CFG.budgets if b <= B] or [1])
        a, c = eval_policy(md, {q: ("SC", fb) for q in QIDS}); rec(mid,"ref","Fixed-b",B,a,0.0,c)
        rng = np.random.default_rng(CFG.seed)
        lo_b = max([b for b in CFG.budgets if b <= B] or [1]); hi_b = min([b for b in CFG.budgets if b >= B] or [16])
        p_hi = 0.0 if hi_b == lo_b else float(np.clip((B-lo_b)/(hi_b-lo_b),0,1))
        ra = [eval_policy(md, {q: ("SC", hi_b if rng.random() < p_hi else lo_b) for q in QIDS}) for _ in range(5)]
        rec(mid,"ref","Random (budget-matched)",B,float(np.mean([x[0] for x in ra])),
            float(np.std([x[0] for x in ra])), float(np.mean([x[1] for x in ra])))
        a, c = eval_policy(md, O1[B]); rec(mid,"ref","Oracle 1-D (Zhai) [UB]",B,a,0.0,c)
        a, c = eval_policy(md, O2[B]); rec(mid,"ref","Oracle 2-D (ours) [UB]",B,a,0.0,c)

        # S0: Zhai et al. classifier on 1-D action set
        a0,s0,c0,i0,_ = learner_classification(md, ACT_1D, B, False, O1[B])
        rec(mid,"S0","AdaCompute (Zhai et al.)",B,a0,s0,c0, imit_1D=i0)
        # S1: joint action space (clf)
        a1,s1,c1,i1,_ = learner_classification(md, acts2, B, False, O2[B])
        rec(mid,"S1","+ joint action space (clf)",B,a1,s1,c1, imit_2D=i1)
        # S2: surface regression
        a2,s2,c2,i2,ch2,r2,_ = learner_surface(md, acts2, B, False, O2[B])
        rec(mid,"S2","+ utility-surface regression",B,a2,s2,c2, imit_2D=i2, surf_R2=r2)
        # S3: semantic entropy
        a3,s3,c3,i3,ch3,r3,pa3 = learner_surface(md, acts2, B, True, O2[B])
        rec(mid,"S3","+ semantic entropy",B,a3,s3,c3, imit_2D=i3, surf_R2=r3)
        # S5: probe features
        a5,s5,c5,i5,ch5,r5,pa5 = learner_surface(md, acts2, B, True, O2[B], use_probe=True)
        rec(mid,"S5","+ probe features",B,a5,s5,c5, imit_2D=i5, surf_R2=r5)

        # S3b and S5b: budget-only action set
        a3b,s3b,c3b,i1d,_,r3b,_ = learner_surface(md, ACT_1D, B, True, O1[B])
        rec(mid,"S3b","ours, 1-D (budget only) + sem",B,a3b,s3b,c3b, imit_1D=i1d, surf_R2=r3b)
        a5b,s5b,c5b,i5b,_,r5b,_ = learner_surface(md, ACT_1D, B, True, O1[B], use_probe=True)
        rec(mid,"S5b","ours, 1-D (budget only) + probe",B,a5b,s5b,c5b, imit_1D=i5b, surf_R2=r5b)

        # S4: probe reuse + CRC escalation
        Br = B*(1-CFG.reserve)
        _,_,_,_,ch3r,_,pa_r = learner_surface(md, acts2, Br, True, O2[B])
        chE, nE = escalate(md, ch3r, acts2, B, TAU, pa_r)
        a4, c4 = eval_policy(md, chE)
        rec(mid,"S4","+ prefix reuse + CRC escalation",B,a4,s3,max(0.0, c4-md["probe_saving"]),
            escalated=nE, tau=TAU)
        redO = oracle_at(md, acts2, find_lambda(md, acts2, Br))
        chO, nO = escalate(md, redO, acts2, B, TAU, md["ACC"])
        aO, cO = eval_policy(md, chO)
        rec(mid,"ref","S4-oracle [UB, not deployable]",B,aO,0.0,
            max(0.0, cO-md["probe_saving"]), escalated=nO)

    SUMMARY[mid] = {"tau":TAU, "alpha_ach":AACH, "auroc":AUROC, "headroom":HR,
                    "pareto":[f"{a}@{b}" for a,b in md["PARETO"]],
                    "probe_saving":md["probe_saving"],
                    "sem_r": float(np.corrcoef([md["PROBE"][q]["sem_ent"] for q in QIDS],
                                               [md["ACC"][(q,"SC",1)] for q in QIDS])[0,1]),
                    "tok_r": float(np.corrcoef([md["PROBE"][q]["tok_ent"] for q in QIDS],
                                               [md["ACC"][(q,"SC",1)] for q in QIDS])[0,1])}
DF = pd.DataFrame(ROWS)
print("\nladder complete for all models")



Qwen2.5-3B-Instruct
  prediction space restricted to Pareto set: 7/12 actions


Validating:           |   0%            

  cost-monotonicity check (Proposition 1): 1-D=True, 2-D=True


Validating:           |   0%            

  CRC n=1600 tau=0.950 alpha_ach=0.348 AUROC=0.933
  [backup] saved Phi-3.5-mini-instruct_200q_probe.pkl

--- B = 1.5 (oracle headroom +3.33 pp) ---
  ref  Fixed-b                          B=1.5  acc=0.4913±0.000  cost=1.00  
  ref  Random (budget-matched)          B=1.5  acc=0.5129±0.004  cost=1.52  
  ref  Oracle 1-D (Zhai) [UB]           B=1.5  acc=0.5606±0.000  cost=1.50  
  ref  Oracle 2-D (ours) [UB]           B=1.5  acc=0.5940±0.000  cost=1.49  
  [backup] saved Phi-3.5-mini-instruct_200q.pkl
  S0   AdaCompute (Zhai et al.)         B=1.5  acc=0.4961±0.002  cost=1.21  imit_1D=0.643
  S1   + joint action space (clf)       B=1.5  acc=0.5004±0.004  cost=1.20  imit_2D=0.625
  S2   + utility-surface regression     B=1.5  acc=0.5301±0.004  cost=1.50  imit_2D=0.522  surf_R2=0.463
  S3   + semantic entropy               B=1.5  acc=0.5273±0.004  cost=1.50  imit_2D=0.543  surf_R2=0.539
  S5   + probe features                 B=1.5  acc=0.5262±0.005  cost=1.49  imit_2D=0.536  surf_R2=0.631


## 6. Empirical Results: Tables, Visualizations, and Artifact Archival

Generates publication tables, comparison figures, and archives all empirical artifacts.


In [ ]:
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":200,"font.size":9,"axes.grid":True,
    "grid.alpha":0.25,"axes.spines.top":False,"axes.spines.right":False,"figure.autolayout":True})
P = {"ours":"#1b3a6b","base":"#8c8c8c","hi":"#c0392b","ok":"#1e8449","gold":"#b8860b"}
def show(f, n):
    f.savefig(f"{CFG.out}/figs/{n}.png", bbox_inches="tight")
    f.savefig(f"{CFG.out}/figs/{n}.pdf", bbox_inches="tight")
    plt.show(); plt.close(f)

# -----------------------------------------------------------------------------
# Publication Result Tables (Tables 1-5)
# -----------------------------------------------------------------------------

print("="*85 + "\nTABLE 1 — Budget-only action set, accuracy averaged over B in {1.5, 2.0, 3.0}\n" + "="*85)
t1_rows = []
for mid in MODELS:
    sh = mid.split("/")[-1]
    d = DF[DF.Model == sh]
    fb = d[d.Method == "Fixed-b"]["Acc"].mean()
    rnd = d[d.Method == "Random (budget-matched)"]["Acc"].mean()
    s0 = d[d.Stage == "S0"]["Acc"].mean()
    s5b = d[d.Stage == "S5b"]["Acc"].mean()
    o1 = d[d.Method == "Oracle 1-D (Zhai) [UB]"]["Acc"].mean()
    gain = 100 * (s5b - s0)
    t1_rows.append({"Solver": sh, "Fixed-b": fb, "Random": rnd, "S0": s0,
                    "S5b (ours)": s5b, "1-D oracle": o1, "Gain (pp)": gain})
T1 = pd.DataFrame(t1_rows)
print(T1.round(4).to_string(index=False))
T1.to_csv(f"{CFG.out}/table1_budget_only.csv", index=False)

print("\n" + "="*85 + "\nTABLE 2 — Accuracy by stage, averaged over budgets; R2 at S2 / S3 / S5\n" + "="*85)
t2_rows = []
for mid in MODELS:
    sh = mid.split("/")[-1]
    d = DF[DF.Model == sh]
    s0 = d[d.Stage == "S0"]["Acc"].mean()
    s1 = d[d.Stage == "S1"]["Acc"].mean()
    s2 = d[d.Stage == "S2"]["Acc"].mean()
    s3 = d[d.Stage == "S3"]["Acc"].mean()
    s5 = d[d.Stage == "S5"]["Acc"].mean()
    s5b = d[d.Stage == "S5b"]["Acc"].mean()
    r2_s2 = d[d.Stage == "S2"]["surf_R2"].mean()
    r2_s3 = d[d.Stage == "S3"]["surf_R2"].mean()
    r2_s5 = d[d.Stage == "S5"]["surf_R2"].mean()
    t2_rows.append({"Solver": sh, "S0": s0, "S1": s1, "S2": s2, "S3": s3, "S5": s5, "S5b": s5b,
                    "R2 S2 / S3 / S5": f"{r2_s2:.2f} / {r2_s3:.2f} / {r2_s5:.2f}"})
T2 = pd.DataFrame(t2_rows)
print(T2.to_string(index=False))
T2.to_csv(f"{CFG.out}/table2_stages.csv", index=False)

print("\n" + "="*85 + "\nTABLE 3 — Oracle accuracy with budget-only (1-D) and joint (2-D) action sets\n" + "="*85)
t3_rows = []
for mid in MODELS:
    sh = mid.split("/")[-1]
    d = DF[DF.Model == sh]
    for B in CFG.targets:
        o1 = d[(d.B == B) & (d.Method == "Oracle 1-D (Zhai) [UB]")]["Acc"].iloc[0]
        o2 = d[(d.B == B) & (d.Method == "Oracle 2-D (ours) [UB]")]["Acc"].iloc[0]
        t3_rows.append({"Solver": sh, "B": B, "1-D oracle": o1, "2-D oracle": o2,
                        "Headroom (pp)": 100 * (o2 - o1)})
T3 = pd.DataFrame(t3_rows)
print(T3.round(4).to_string(index=False))
T3.to_csv(f"{CFG.out}/table3_oracles.csv", index=False)

print("\n" + "="*85 + "\nTABLE 4 — Surviving actions, verifier AUROC, and oracle headroom\n" + "="*85)
t4_rows = []
for mid in MODELS:
    sh = mid.split("/")[-1]; md = MD[mid]
    bon_surv = [f"b={b}" for (top, b) in md["PARETO"] if top == "BoN"]
    tree_surv = [f"{top}@{b}" for (top, b) in md["PARETO"] if top in ("ToT", "FoT")]
    hr_avg = 100 * np.mean(list(SUMMARY[mid]["headroom"].values()))
    s5_avg = DF[(DF.Model == sh) & (DF.Stage == "S5")]["Acc"].mean()
    s5b_avg = DF[(DF.Model == sh) & (DF.Stage == "S5b")]["Acc"].mean()
    t4_rows.append({"Solver": sh,
                    "Best-of-N": ", ".join(bon_surv) if bon_surv else "none",
                    "Tree": ", ".join(tree_surv) if tree_surv else "none",
                    "Verifier AUROC": SUMMARY[mid]["auroc"],
                    "Headroom (pp)": hr_avg,
                    "Learned topology gain (pp)": 100 * (s5_avg - s5b_avg)})
T4 = pd.DataFrame(t4_rows)
print(T4.round(3).to_string(index=False))
T4.to_csv(f"{CFG.out}/table4_surviving_actions.csv", index=False)

print("\n" + "="*85 + "\nTABLE 5 — Correlation with single-pass accuracy and Surface R2\n" + "="*85)
t5_rows = []
for mid in MODELS:
    sh = mid.split("/")[-1]
    d = DF[DF.Model == sh]
    r2_txt = d[d.Stage == "S2"]["surf_R2"].mean()
    r2_sem = d[d.Stage == "S3"]["surf_R2"].mean()
    t5_rows.append({"Solver": sh,
                    "token H": SUMMARY[mid]["tok_r"],
                    "semantic H": SUMMARY[mid]["sem_r"],
                    "text features R2": r2_txt,
                    "+Hsem R2": r2_sem})
T5 = pd.DataFrame(t5_rows)
print(T5.round(3).to_string(index=False))
T5.to_csv(f"{CFG.out}/table5_entropy_signals.csv", index=False)

# -----------------------------------------------------------------------------
# Publication Figures (Figures 4-7)
# -----------------------------------------------------------------------------

# ── Fig 4 of paper: Stages against both oracles ──
STAGE_ORDER = ["S0", "S1", "S2", "S3", "S5"]
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.8*len(MODELS), 3.8), squeeze=False)
for k, mid in enumerate(MODELS):
    sh = mid.split("/")[-1]; ax = axes[0][k]
    y_stages = [DF[(DF.Model == sh) & (DF.Stage == s)]["Acc"].mean() for s in STAGE_ORDER]
    o1_mean = DF[(DF.Model == sh) & (DF.Method == "Oracle 1-D (Zhai) [UB]")]["Acc"].mean()
    o2_mean = DF[(DF.Model == sh) & (DF.Method == "Oracle 2-D (ours) [UB]")]["Acc"].mean()
    s5b_mean = DF[(DF.Model == sh) & (DF.Stage == "S5b")]["Acc"].mean()

    ax.plot(range(len(STAGE_ORDER)), y_stages, marker="o", lw=1.8, color="#1b3a6b", label="joint action set")
    ax.axhline(o2_mean, ls="--", color="#4a86b8", lw=1.2, label="2-D oracle")
    ax.axhline(o1_mean, ls=":", color="#2e6b9e", lw=1.2, label="1-D oracle")
    ax.scatter([4], [s5b_mean], marker="s", color="#c0392b", s=60, zorder=5, label="S5b (budget only)")

    ax.set_xticks(range(len(STAGE_ORDER)))
    ax.set_xticklabels(STAGE_ORDER, fontsize=9)
    ax.set_title(sh, fontsize=10, fontweight="bold", loc="left")
    ax.set_xlabel("pipeline stage")
    if k == 0:
        ax.set_ylabel("accuracy")
        ax.legend(frameon=False, fontsize=7.5, loc="lower right")
fig.suptitle("Figure 4: Stages against both oracles (dashed=2-D oracle, dotted=1-D oracle, red square=S5b)",
             fontweight="bold", x=0.01, ha="left", fontsize=10)
show(fig, "fig4_stages_oracles")

# ── Fig 5 of paper: Oracle headroom by solver and target budget ──
fig, ax = plt.subplots(figsize=(6.8, 3.8))
w = 0.25; x = np.arange(len(MODELS))
colors_b = ["#a8c5da", "#4a86b8", "#0d3b66"]
for j, B in enumerate(CFG.targets):
    vals = [100 * SUMMARY[m]["headroom"][B] for m in MODELS]
    bars = ax.bar(x + (j-1)*w, vals, w, label=f"B = {B}", color=colors_b[j % len(colors_b)])
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.1f}", xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 2 if h >= 0 else -10), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels([m.split("/")[-1] for m in MODELS], fontsize=9)
ax.set_ylabel("oracle headroom (accuracy points)")
ax.legend(frameon=False, fontsize=8, loc="upper right")
ax.set_title("Figure 5: Oracle headroom of the joint action set over the budget-only set",
             fontweight="bold", loc="left", fontsize=10)
show(fig, "fig5_headroom")

# ── Fig 6 of paper: Difficulty signals & Surface R2 progression ──
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(9.5, 3.8))
x = np.arange(len(MODELS)); w = 0.35
# Left: |r| with single-pass accuracy
ax_left.bar(x - w/2, [abs(SUMMARY[m]["tok_r"]) for m in MODELS], w,
            label="token entropy", color="#a8c5da")
ax_left.bar(x + w/2, [abs(SUMMARY[m]["sem_r"]) for m in MODELS], w,
            label="semantic entropy", color="#c0392b")
for i, m in enumerate(MODELS):
    ax_left.text(i - w/2, abs(SUMMARY[m]["tok_r"]) + 0.02, f"{abs(SUMMARY[m]['tok_r']):.2f}",
                 ha="center", fontsize=7.5)
    ax_left.text(i + w/2, abs(SUMMARY[m]["sem_r"]) + 0.02, f"{abs(SUMMARY[m]['sem_r']):.2f}",
                 ha="center", fontsize=7.5)
ax_left.set_xticks(x); ax_left.set_xticklabels([m.split("/")[-1] for m in MODELS], fontsize=8, rotation=10)
ax_left.set_ylabel("|r| with single-pass accuracy")
ax_left.set_ylim(0, 1.0)
ax_left.legend(frameon=False, fontsize=8)
ax_left.set_title("Correlation with single-pass accuracy", loc="left", fontweight="bold", fontsize=9.5)

# Right: utility-surface R2
w3 = 0.26
r2_txt = [DF[(DF.Model == m.split("/")[-1]) & (DF.Stage == "S2")]["surf_R2"].mean() for m in MODELS]
r2_sem = [DF[(DF.Model == m.split("/")[-1]) & (DF.Stage == "S3")]["surf_R2"].mean() for m in MODELS]
r2_prb = [DF[(DF.Model == m.split("/")[-1]) & (DF.Stage == "S5")]["surf_R2"].mean() for m in MODELS]
ax_right.bar(x - w3, r2_txt, w3, label="16 text features", color="#a8c5da")
ax_right.bar(x, r2_sem, w3, label="+ semantic entropy", color="#2e7d32")
ax_right.bar(x + w3, r2_prb, w3, label="+ probe features", color="#0d3b66")
for i in range(len(MODELS)):
    ax_right.text(i - w3, max(0, r2_txt[i]) + 0.02, f"{r2_txt[i]:.2f}", ha="center", fontsize=7)
    ax_right.text(i, max(0, r2_sem[i]) + 0.02, f"{r2_sem[i]:.2f}", ha="center", fontsize=7)
    ax_right.text(i + w3, max(0, r2_prb[i]) + 0.02, f"{r2_prb[i]:.2f}", ha="center", fontsize=7)
ax_right.set_xticks(x); ax_right.set_xticklabels([m.split("/")[-1] for m in MODELS], fontsize=8, rotation=10)
ax_right.set_ylabel("utility-surface R^2 (out-of-fold)")
ax_right.set_ylim(0, 0.75)
ax_right.legend(frameon=False, fontsize=7.5)
ax_right.set_title("Utility-surface R^2 progression", loc="left", fontweight="bold", fontsize=9.5)

fig.suptitle("Figure 6: Left: correlation of entropy features. Right: surface R2 progression.",
             fontweight="bold", x=0.01, ha="left", fontsize=10)
show(fig, "fig6_signals_and_r2")

# ── Fig 7 of paper: action dominance in the MEASURED cost/accuracy plane ──
COSTS = pd.read_csv(f"{CFG.out}/costs.csv")
_models = list(dict.fromkeys(COSTS["model"]))
fig, axes = plt.subplots(1, len(_models), figsize=(4.4*len(_models), 3.4))
axes = np.atleast_1d(axes)
for ax, m in zip(axes, _models):
    d = COSTS[COSTS.model == m]; on, off = d[d.pareto], d[~d.pareto]
    fr = on.sort_values("cost")
    ax.plot(fr["cost"], fr["acc"], ls="--", lw=1.1, color=P["ours"], alpha=.6, zorder=1)
    ax.scatter(on["cost"], on["acc"], s=60, color=P["ours"], zorder=3, label="Pareto-optimal")
    ax.scatter(off["cost"], off["acc"], s=48, marker="x", color=P["hi"],
               linewidth=1.5, zorder=3, label="dominated")
    for _, r in d.iterrows():
        if r.topology in ("ToT","FoT") or r.action == "BoN@2":
            ax.annotate(r.action, (r.cost, r.acc), fontsize=7, xytext=(4,4),
                        textcoords="offset points", color=P["ours"] if r.pareto else P["hi"])
    ax.set_xscale("log", base=2); ax.set_xticks([1,2,4,8,16]); ax.set_xticklabels([1,2,4,8,16])
    ax.set_xlabel("measured cost (sample-equivalents)"); ax.set_title(m, fontsize=10, loc="left")
axes[0].set_ylabel("mean accuracy"); axes[0].legend(frameon=False, fontsize=8, loc="lower right")
show(fig, "fig7_dominance")

HEAD = {"profile":PROFILE, "n_questions":len(ITEMS), "models":MODELS,
        "n_actions_full":len(ACTIONS), "targets":list(CFG.targets),
        "per_model":SUMMARY,
        "throughput_tok_s":STATS["tok"]/max(STATS["sec"],1),
        "wall_min":(time.time()-T_BOOT)/60}
json.dump(HEAD, open(f"{CFG.out}/headline.json","w"), indent=2, default=str)
DF.to_csv(f"{CFG.out}/ladder.csv", index=False)
import shutil
_zip_path = shutil.make_archive(os.path.join(os.path.dirname(CFG.out) or ".", "artifacts"), "zip", CFG.out)
print(f"\nartifacts -> {_zip_path}  |  {thr()}")

# Synchronize generated evaluation artifacts with Hugging Face repository
if SYNC:
    print("\nuploading all research artifacts to HF backup...")
    _uploaded = 0
    for root, dirs, files in os.walk(CFG.out):
        dirs[:] = [d for d in dirs if d not in ("_hf_pull", "__pycache__")]
        for fname in sorted(files):
            if fname.endswith(".tmp"): continue
            local = os.path.join(root, fname)
            remote = os.path.relpath(local, CFG.out)
            try:
                _hf.upload_file(path_or_fileobj=local,
                                path_in_repo=remote,
                                repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN,
                                commit_message=f"artifact {remote}")
                _uploaded += 1
            except Exception as e:
                print(f"  upload failed: {remote} -> {e}")
    # also upload the zip
    try:
        _hf.upload_file(path_or_fileobj=_zip_path,
                        path_in_repo="artifacts.zip",
                        repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN,
                        commit_message="artifacts.zip")
        _uploaded += 1
    except Exception: pass
    print(f"  {_uploaded} files uploaded to huggingface.co/datasets/{HF_REPO}")
else:
    print("backup OFF: artifacts not uploaded")


TABLE 1 — Budget-only action set, accuracy averaged over B in {1.5, 2.0, 3.0}
               Solver  Fixed-b  Random     S0  S5b (ours)  1-D oracle  Gain (pp)
  Qwen2.5-3B-Instruct   0.5175  0.5329 0.5024      0.5425      0.5934     4.0188
Phi-3.5-mini-instruct   0.5351  0.5495 0.5347      0.5578      0.6145     2.3146

TABLE 2 — Accuracy by stage, averaged over budgets; R2 at S2 / S3 / S5
               Solver       S0       S1       S2       S3       S5      S5b    R2 S2 / S3 / S5
  Qwen2.5-3B-Instruct 0.502354 0.511917 0.549660 0.557076 0.556889 0.542542 0.46 / 0.54 / 0.63
Phi-3.5-mini-instruct 0.534667 0.533375 0.560222 0.557861 0.555861 0.557813 0.21 / 0.26 / 0.46

TABLE 3 — Oracle accuracy with budget-only (1-D) and joint (2-D) action sets
               Solver   B  1-D oracle  2-D oracle  Headroom (pp)
  Qwen2.5-3B-Instruct 1.5      0.5606      0.5940         3.3333
  Qwen2.5-3B-Instruct 2.0      0.5909      0.6339         4.2917
  Qwen2.5-3B-Instruct 3.0      0.6288      0.6614

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


...

Validating:           |   0%            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

  37 files uploaded to huggingface.co/datasets/shkshreyas/reason-x-ckpt


---
## 7. Discussion & Methodological Analysis

### Key Empirical Takeaways
1. **Learner Gain Isolation ($S1 \to S2$):** Direct utility-surface regression significantly outperforms discrete classification by preserving ordinal ranking and density across actions.
2. **Action Space Expressivity ($S5b \to S5$):** Expanding the action space from budget-only selection to joint topology-budget selection raises the achievable Pareto frontier.
3. **Statistical Power & Generalization:** Evaluating policies on dual reasoning benchmarks (MATH-500 and GSM8K) with paired bootstrap resampling establishes statistically robust confidence intervals.


In [ ]:
# Paired bootstrap confidence intervals and Figure 3 of the paper
N_BOOT = 10_000
rng_b = np.random.default_rng(CFG.seed)
SEEDS = list(range(CFG.n_seeds))
t_bs = time.time()

def per_q(md, chosen):
    return np.array([md["ACC"][(q,) + chosen[q]] for q in QIDS], dtype=float)

PERQ = {}          # model -> {"S0","S5b","S5"} -> per-question accuracy, mean over seeds & budgets
for mid in MODELS:
    md = MD[mid]; short = mid.split("/")[-1]
    acts2 = md["PARETO"] if len(md["PARETO"]) >= 3 else ACTIONS
    acc = {k: np.zeros(len(QIDS)) for k in ("S0", "S5b", "S5")}
    for B in CFG.targets:
        O1 = oracle_at(md, ACT_1D, find_lambda(md, ACT_1D, B))
        O2 = oracle_at(md, acts2,  find_lambda(md, acts2,  B))
        for sd in SEEDS:
            ch0  = learner_classification(md, ACT_1D, B, False, O1, seeds=[sd])[4]
            ch5b = learner_surface(md, ACT_1D, B, True, O1, seeds=[sd], use_probe=True)[4]
            ch5  = learner_surface(md, acts2,  B, True, O2, seeds=[sd], use_probe=True)[4]
            acc["S0"]  += per_q(md, ch0)
            acc["S5b"] += per_q(md, ch5b)
            acc["S5"]  += per_q(md, ch5)
    k = len(CFG.targets) * len(SEEDS)
    PERQ[short] = {s: v / k for s, v in acc.items()}
    print(f"  {short}: re-fitted {3*k} allocations [{(time.time()-t_bs)/60:.1f} min]")

def boot(diff):
    """Mean, 95% percentile CI and two-sided p-value of a paired per-question gain."""
    n = len(diff)
    idx = rng_b.integers(0, n, size=(N_BOOT, n))
    means = diff[idx].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    p = 2 * min((means <= 0).mean(), (means >= 0).mean())
    return 100*diff.mean(), 100*lo, 100*hi, max(p, 1/N_BOOT)

ROWS = []
for short, d in PERQ.items():
    for name, a, b in (("learner", "S5b", "S0"), ("topology", "S5", "S5b")):
        m, lo, hi, p = boot(d[a] - d[b])
        ROWS.append({"model": short, "gain": name, "mean_pp": m, "ci_lo": lo, "ci_hi": hi, "p": p})

# Pooled estimation: joint bootstrap resampling across question clusters
for name, a, b in (("learner", "S5b", "S0"), ("topology", "S5", "S5b")):
    D = np.mean([PERQ[s][a] - PERQ[s][b] for s in PERQ], axis=0)
    m, lo, hi, p = boot(D)
    ROWS.append({"model": f"pooled ({len(PERQ)} models)", "gain": name, "mean_pp": m, "ci_lo": lo, "ci_hi": hi, "p": p})

BOOT = pd.DataFrame(ROWS)
BOOT.to_csv(f"{CFG.out}/bootstrap_ci.csv", index=False)

print(f"\n== PAIRED BOOTSTRAP, {N_BOOT:,} resamples over {len(QIDS)} questions (accuracy points) ==")
for name in ("learner", "topology"):
    print(f"\n  {name.upper()} GAIN  " + ("(S5b - S0)" if name == "learner" else "(S5 - S5b)"))
    print(f"  {'model':<28}{'mean':>8}{'95% CI':>20}{'p':>10}   ")
    for _, r in BOOT[BOOT.gain == name].iterrows():
        sig = "significant" if r.ci_lo > 0 or r.ci_hi < 0 else "n.s."
        print(f"  {r.model:<28}{r.mean_pp:+8.2f}   [{r.ci_lo:+6.2f}, {r.ci_hi:+6.2f}]{r.p:10.4f}   {sig}")

# Figure 3 of the paper (Two-panel paired bootstrap 95% CIs)
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(9.0, 4.2), sharey=True)

model_order = [m.split("/")[-1] for m in MODELS] + [f"pooled ({len(PERQ)} models)"]
y_pos = list(range(len(model_order)))[::-1]

for ax, gain_type, col, title in [
    (ax_l, "learner", "#1b3a6b", "allocator gain (S5b - S0)"),
    (ax_r, "topology", "#c0392b", "topology gain (S5 - S5b)")
]:
    sub = BOOT[BOOT.gain == gain_type].set_index("model")
    for y, m in zip(y_pos, model_order):
        if m in sub.index:
            r = sub.loc[m]
            ax.errorbar(r.mean_pp, y, xerr=[[r.mean_pp - r.ci_lo], [r.ci_hi - r.mean_pp]],
                        fmt="o", color=col, capsize=3.5, lw=1.6, ms=5.5)
    ax.axvline(0, color="k", lw=0.8, ls="-")
    ax.set_title(title, fontsize=9.5, fontweight="bold")
    ax.set_xlabel("accuracy points, 95% CI", fontsize=8.5)
    ax.set_xlim(-5, 7)

ax_l.set_yticks(y_pos)
ax_l.set_yticklabels(model_order, fontsize=8.5)

fig.suptitle("Figure 3: Paired-bootstrap 95% intervals. Left: allocator gain. Right: topology gain.",
             fontweight="bold", x=0.01, ha="left", fontsize=10)
fig.tight_layout()
fig.savefig(f"{CFG.out}/figs/fig3_bootstrap.pdf", bbox_inches="tight")
fig.savefig(f"{CFG.out}/figs/fig3_bootstrap.png", dpi=200, bbox_inches="tight")
plt.show(); plt.close(fig)

import shutil
_zip_path = shutil.make_archive(os.path.join(os.path.dirname(CFG.out) or ".", "artifacts"), "zip", CFG.out)
print(f"\nsaved bootstrap_ci.csv and figs/fig3_bootstrap.pdf | artifacts.zip refreshed | "
      f"{(time.time()-t_bs)/60:.1f} min")

# Export per-question accuracy matrix for downstream statistical analysis
pq = pd.DataFrame({"qid": QIDS})
for short, d in PERQ.items():
    for k_, v_ in d.items(): pq[f"{short}|{k_}"] = v_
pq.to_csv(f"{CFG.out}/perq_accuracy.csv", index=False)

if SYNC:
    for f in ["bootstrap_ci.csv", "perq_accuracy.csv", "figs/fig3_bootstrap.pdf", "figs/fig3_bootstrap.png"]:
        try:
            _hf.upload_file(path_or_fileobj=f"{CFG.out}/{f}", path_in_repo=f, repo_id=HF_REPO,
                            repo_type="dataset", token=HF_TOKEN, commit_message=f"artifact {f}")
            print(f"  uploaded {f}")
        except Exception as e:
            print(f"  upload FAILED for {f}: {e}")


  Qwen2.5-3B-Instruct: re-fitted 45 allocations [3.0 min]
  Phi-3.5-mini-instruct: re-fitted 45 allocations [6.0 min]

== PAIRED BOOTSTRAP, 10,000 resamples over 200 questions (accuracy points) ==

  LEARNER GAIN  (S5b - S0)
  model                           mean              95% CI         p   
  Qwen2.5-3B-Instruct            +4.02   [ +2.71,  +5.44]    0.0001   significant
  Phi-3.5-mini-instruct          +2.31   [ +1.39,  +3.28]    0.0001   significant
  pooled (2 models)              +3.17   [ +2.31,  +4.10]    0.0001   significant

  TOPOLOGY GAIN  (S5 - S5b)
  model                           mean              95% CI         p   
  Qwen2.5-3B-Instruct            +1.43   [ +0.20,  +2.84]    0.0212   significant
  Phi-3.5-mini-instruct          -0.20   [ -1.73,  +1.33]    0.7990   n.s.
  pooled (2 models)              +0.62   [ -0.42,  +1.65]    0.2338   n.s.

saved bootstrap_ci.csv and figs/fig3_bootstrap.pdf | artifacts.zip refreshed | 6.0 min
  uploaded bootstrap_ci.csv
  upload

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

  uploaded figs/fig3_bootstrap.png
